# Literature Research Search

This notebook loads ArXiv metadata, creates text embeddings with the Harrier model, and searches the saved index for semantically similar papers.

It is organized into three parts:
1. Loading the dataset and building the vector index in the RAM. This is done in the first cell. It will take some time. This only needs to be done once.
2. Running a semantic search query against the saved embeddings. This is done in the second cell. The query can be adapted. According to the query the correct prompt_name in the encoding has to be chosen.
3. Exporting the matching results to CSV and Parquet files. This is done in the second cell.

In [3]:
from pathlib import Path

import duckdb
import torch
from sentence_transformers import SentenceTransformer
from usearch.index import Index

# File paths for the JSON and Parquet files.
parquet_file = Path("dataset/arxiv_metadata.parquet")
# Model name for the embedding model.
model_name = "microsoft/harrier-oss-v1-0.6b"
# Path prefix for saving embeddings and metadata.
output_prefix = Path(f"output_data/{model_name.replace('/', '_')}")
# Column name for the unique identifier in the Parquet file.
id_column = "id"
# Columns to be merged into a single text input for embedding.
text_columns = ["title", "abstract"]


def read_rows(limit: int | None = None, offset: int = 0) -> list[tuple]:
    """Read selected rows from the Parquet file using DuckDB."""

    selected_columns = [id_column, *text_columns]
    columns_sql = ", ".join(f'"{column}"' for column in selected_columns)
    limit_sql = ""

    if limit is not None:
        limit_sql = f"LIMIT {limit}"

    query = f"""
        SELECT {columns_sql}
        FROM read_parquet('{parquet_file.as_posix()}')
        {limit_sql}
        OFFSET {offset}
    """

    return duckdb.sql(query).fetchall()


def build_ids_and_texts(rows: list[tuple]) -> tuple[list[str], list[str]]:
    """Transform raw Parquet rows into IDs and combined text values."""

    selected_columns = [id_column, *text_columns]
    ids = []
    texts = []

    for row in rows:
        row_values = dict(zip(selected_columns, row, strict=True))

        ids.append(str(row_values[id_column]))

        parts = []
        for column in text_columns:
            value = row_values.get(column)

            if value is not None:
                # Remove excessive whitespace and newlines from the text
                value = " ".join(str(value).split())

                if value:
                    parts.append(value)

        texts.append(". ".join(parts))

    return ids, texts


vector_index_path = output_prefix.with_name(f"{output_prefix.stem}_usearch.index")

rows = read_rows()
ids, _ = build_ids_and_texts(rows)

device = "cuda" if torch.cuda.is_available() else "cpu"
if device == "cuda":
    print(f"Using GPU: {torch.cuda.get_device_name(0)}")
else:
    print("CUDA not available, using CPU.")

model = SentenceTransformer(
    "microsoft/harrier-oss-v1-0.6b",
    model_kwargs={"dtype": "auto"},
    device=device,
)

index = Index(
    ndim=model.get_embedding_dimension(),
    metric="cos",
    dtype="f32",
)
index.load(str(vector_index_path))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

CUDA not available, using CPU.


Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

In [ ]:
import numpy as np

# Search saved embeddings with a text query using cosine similarity and saves
# the results in a parquet and csv file with the matching parquet rows.

# Query string for searching similar papers in the embedded dataset.
query = (
    "What deepfake detection tools are used by professional journalists and "
    "fact checkers, to verify the authenticity of digital media?"
)
# Number of top similar results to retrieve. Set to None to ignore the top_k limit.
top_k = 1000


def save_matching_results(
    results: list[dict],
    output_parquet_path: Path,
    output_csv_path: Path,
    query: str,
) -> None:
    """Save full matching rows to Parquet and CSV using DuckDB."""

    if not results:
        print("No matching results found. Nothing will be saved.")
        return

    con = duckdb.connect()

    try:
        con.execute(
            """
            CREATE TEMP TABLE search_results (
                id VARCHAR,
                similarity_score DOUBLE,
                search_query VARCHAR
            )
            """
        )

        con.executemany(
            """
            INSERT INTO search_results VALUES (?, ?, ?)
            """,
            [(result["id"], result["similarity_score"], query) for result in results],
        )

        # Add after p.* to remove \n and \r characters from title and abstract
        # columns
        # REPLACE(REPLACE(CAST(title AS VARCHAR),
        # chr(13), ' '), chr(10), ' ') AS title,
        # REPLACE(REPLACE(CAST(abstract AS VARCHAR),
        # chr(13), ' '), chr(10), ' ') AS abstract,
        sql_query = f"""
            SELECT
                p.*,
                s.similarity_score,
                s.search_query
            FROM read_parquet('{parquet_file.as_posix()}') AS p
            JOIN search_results AS s
                ON CAST(p.{id_column} AS VARCHAR) = s.id
            ORDER BY s.similarity_score DESC
        """

        con.execute(
            f"COPY ({sql_query}) TO '{output_parquet_path.as_posix()}' (FORMAT PARQUET)"
        )
        con.execute(
            f"""
            COPY (
                {sql_query}
            )
            TO '{output_csv_path.as_posix()}'
            (
                FORMAT CSV,
                HEADER TRUE,
                DELIMITER ';'
            )
            """
        )
    finally:
        con.close()

    print(f"Saved full matching rows to {output_parquet_path}")
    print(f"Saved full matching rows to {output_csv_path}\n")


print(f"Start searching for similar results for query:\n {query}\n")

# "web_search_query": "Instruct: Given a web search query, retrieve relevant
# passages that answer the query\nQuery: ",
# "sts_query": "Instruct: Retrieve semantically similar text\nQuery: ",
# "bitext_query": "Instruct: Retrieve parallel sentences\nQuery: "
query_embedding = model.encode(
    [query],
    normalize_embeddings=True,
    prompt_name="web_search_query",
)[0].astype(np.float32)

# USearch needs a fixed number of candidates.
matches = index.search(query_embedding, top_k)

results = []

for match in matches:
    index_position = int(match.key)

    # With cosine distance: smaller distance = more similar.
    similarity_score = float(1 - match.distance)

    results.append(
        {
            "similarity_score": similarity_score,
            "id": ids[index_position],
        }
    )

results = sorted(
    results,
    key=lambda result: result["similarity_score"],
    reverse=True,
)

output_path = Path(f"{output_prefix}_{query[:20].replace('.', '_')}.json")
output_path.parent.mkdir(parents=True, exist_ok=True)

output_csv_path = output_path.with_suffix(".csv")
output_parquet_path = output_path.with_suffix(".parquet")
save_matching_results(results, output_parquet_path, output_csv_path, query=query)

print("Finished searching for similar results.\n")

Start searching for similar results for query:
 What deepfake detection tools are used by professional journalists and fact checkers, to verify the authenticity of digital media?



FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Saved full matching rows to output_data\microsoft_harrier-oss-v1-0.6b_What deepfake detect.parquet
Saved full matching rows to output_data\microsoft_harrier-oss-v1-0.6b_What deepfake detect.csv

Finished searching for similar results.

